# Baseline Convolutional Autoencoder — Anomaly Detection

This notebook implements a **baseline unsupervised anomaly detector** for machine sound, using a Convolutional Autoencoder (CAE) trained on the DCASE dataset.

**Core idea:** Train exclusively on *normal* recordings. The model learns to reconstruct normal sounds accurately. At inference time, anomalous sounds produce a high reconstruction error — this error is used directly as the anomaly score.

**Pipeline:**
- **Step 0** — Define the CAE architecture
- **Step 1** — Set configuration (paths & hyperparameters)
- **Step 2** — Load WAV files and build a spectrogram DataLoader
- **Step 3** — Train the autoencoder on normal data
- **Step 4** — Score samples and flag anomalies


## Step 0 — Model Architecture

The `ConvAutoencoder` is a **Convolutional Autoencoder (CAE)** that operates on 128×128 single-channel spectrogram images.

**Encoder** — three strided `Conv2d` layers progressively compress the input:

$$
(1, 128, 128) \xrightarrow{\text{stride 2}} (32, 64, 64) \xrightarrow{\text{stride 2}} (64, 32, 32) \xrightarrow{\text{stride 2}} (128, 16, 16)
$$

**Decoder** — three `ConvTranspose2d` layers mirror the encoder and upsample back to the original shape:

$$
(128, 16, 16) \xrightarrow{\times 2} (64, 32, 32) \xrightarrow{\times 2} (32, 64, 64) \xrightarrow{\times 2} (1, 128, 128)
$$

Two utility functions are also defined:
- `train_step` — forward pass + MSE loss between input `x` and reconstruction `x̂`
- `anomaly_score` — returns the per-sample pixel-wise MSE as a scalar anomaly score


In [31]:
import torch
import torch.nn as nn

class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),  # (128x128 → 64x64)
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), # (64x64 → 32x32)
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU()
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat


# Loss
criterion = nn.MSELoss()

# Training step
def train_step(model, x):
    x_hat = model(x)
    loss = criterion(x_hat, x)
    return loss


# Inference (anomaly score)
def anomaly_score(model, x):
    with torch.no_grad():
        x_hat = model(x)
        score = torch.mean((x - x_hat) ** 2, dim=[1,2,3])
    return score

## Step 1 — Configuration

All paths and hyperparameters are defined here. Change `DATA_ROOT` and `MACHINE_TYPE` to point to a different machine's dataset.

| Parameter | Value | Description |
|---|---|---|
| `DATA_ROOT` | path | Root folder of the dataset |
| `MACHINE_TYPE` | `"bearing"` | Machine subfolder name |
| `TARGET_SR` | 16000 | Audio sample rate (Hz) |
| `DURATION` | 1.0 | Clip length in seconds |
| `N_MELS` | 128 | Mel filterbank bins |
| `SPEC_SIZE` | 128 | Spectrogram image size (px) |
| `BATCH_SIZE` | 16 | Training batch size |
| `EPOCHS` | 20 | Number of training epochs |
| `LR` | 1e-3 | Adam learning rate |


In [32]:
import os
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT    = r"C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779"
MACHINE_TYPE = "bearing"   # change to "fan", "valve", etc. for other machines

# ── Audio settings ───────────────────────────────────────────────────────────
TARGET_SR  = 16_000   # resample all audio to 16 kHz
DURATION   = 1.0      # clip length in seconds
N_MELS     = 128      # mel filterbank bins
SPEC_SIZE  = 128      # spectrogram image size fed to the model

# ── Training hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE = 16
EPOCHS     = 20
LR         = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


Device: cpu


## Step 2 — Dataset & DataLoader

`SpectrogramDataset` converts each WAV file into a fixed-size **log-mel spectrogram** image that the model can process:

1. **Load audio** — tries `torchaudio`; falls back to `scipy` for files with unsupported encodings
2. **Mono + resample** — averages stereo channels to mono; resamples to `TARGET_SR` (16 kHz)
3. **Crop / pad** — ensures every clip is exactly `DURATION` seconds (16,000 samples)
4. **Log-mel spectrogram** — applies a mel filterbank then `log(1 + x)` to compress the dynamic range
5. **Resize** — bilinear interpolation to `(SPEC_SIZE × SPEC_SIZE)` so all inputs are the same shape

`collect_wav_files` recursively finds all `.wav` files under the training folder, following the dataset structure `dev_{MACHINE_TYPE}/{MACHINE_TYPE}/train/`.


In [33]:
import numpy as np
from scipy.io import wavfile


class SpectrogramDataset(Dataset):
    """Convert WAV files to fixed-size log-mel spectrograms."""

    def __init__(self, file_paths, target_sr=TARGET_SR, duration=DURATION,
                 n_mels=N_MELS, spec_size=SPEC_SIZE):
        self.file_paths   = file_paths
        self.segment_len  = int(target_sr * duration)
        self.mel          = T.MelSpectrogram(sample_rate=target_sr, n_fft=1024, hop_length=512, n_mels=n_mels)
        self.spec_size    = spec_size
        self.target_sr    = target_sr

    def _load(self, path):
        """Load a WAV file, convert to mono float32, and resample if needed."""
        try:
            waveform, sr = torchaudio.load(path)
        except RuntimeError:
            # Fallback for files that torchaudio cannot decode
            sr, data = wavfile.read(path)
            data = np.asarray(data, dtype=np.float32)
            if np.issubdtype(data.dtype, np.integer):
                data = data / float(np.iinfo(data.dtype).max)
            waveform = torch.from_numpy(data).unsqueeze(0) if data.ndim == 1 \
                       else torch.from_numpy(data.T)
            sr = int(sr)

        if waveform.shape[0] > 1:                          # stereo → mono
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)

        waveform = waveform.squeeze(0)

        # Crop long clips; zero-pad short ones
        if waveform.shape[0] >= self.segment_len:
            waveform = waveform[:self.segment_len]
        else:
            waveform = nn.functional.pad(waveform, (0, self.segment_len - waveform.shape[0]))

        return waveform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        waveform = self._load(self.file_paths[idx])
        spec = self.mel(waveform)                   # (n_mels, time)
        spec = torch.log1p(spec).unsqueeze(0)       # log-compress; add channel dim

        # Resize to (SPEC_SIZE × SPEC_SIZE) so every input is the same shape
        spec = nn.functional.interpolate(
            spec.unsqueeze(0), size=(self.spec_size, self.spec_size),
            mode="bilinear", align_corners=False
        ).squeeze(0)
        return spec


def collect_wav_files(*folders):
    """Recursively collect all .wav paths from the given directories."""
    paths = []
    for folder in folders:
        if os.path.isdir(folder):
            for root, _, files in os.walk(folder):
                for f in files:
                    if f.lower().endswith(".wav"):
                        paths.append(os.path.join(root, f))
    return sorted(paths)


# ── Build DataLoader ─────────────────────────────────────────────────────────
# Expected folder layout: {DATA_ROOT}/dev_{MACHINE_TYPE}/{MACHINE_TYPE}/train/
train_dir   = os.path.join(DATA_ROOT, f"dev_{MACHINE_TYPE}", MACHINE_TYPE, "train")
train_files = collect_wav_files(train_dir)

if len(train_files) == 0:
    print(f"No WAV files found in '{train_dir}'.")
    print("Check DATA_ROOT / MACHINE_TYPE. Using 64 synthetic tensors as fallback.")
    train_loader = DataLoader(
        torch.utils.data.TensorDataset(torch.randn(64, 1, SPEC_SIZE, SPEC_SIZE)),
        batch_size=BATCH_SIZE, shuffle=True
    )
else:
    dataset      = SpectrogramDataset(train_files)
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    print(f"Loaded {len(train_files)} files → {len(train_loader)} batches per epoch")


Loaded 1000 files → 62 batches per epoch


## Step 3 — Training the Autoencoder

The autoencoder is instantiated and trained using the **Adam optimiser** with MSE loss.

Because the training set contains **only normal samples**, the model learns to reconstruct normal bearing sounds accurately. Anomalous sounds will later produce high reconstruction errors because they are outside the distribution the model was trained on.

For each epoch, the mean batch loss is accumulated and printed every 5 epochs so you can monitor convergence.


In [34]:
model     = ConvAutoencoder().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0

    for batch in train_loader:
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        x = x.to(DEVICE)

        optimizer.zero_grad()
        loss = train_step(model, x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch == 1 or epoch % 5 == 0:
        avg = total_loss / len(train_loader)
        print(f"Epoch {epoch:>3}/{EPOCHS}  avg loss = {avg:.6f}")

print("Training complete.")


Epoch   1/20  avg loss = 30.081404
Epoch   5/20  avg loss = 0.456789
Epoch  10/20  avg loss = 0.332951
Epoch  15/20  avg loss = 0.234110
Epoch  20/20  avg loss = 0.210090
Training complete.


## Step 4 — Anomaly Score Evaluation

The trained model is run in evaluation mode over the training set. For each sample the **reconstruction error** (mean squared error between the input and its reconstruction) is recorded as its anomaly score.

A threshold is then chosen at the **95th percentile** of these scores: the top 5 % of samples — those the model struggled most to reconstruct — are flagged as anomalous.

> In a real evaluation you would run this on a separate test set containing both normal and anomalous files and compare the scores against ground-truth labels.


In [35]:
model.eval()
all_scores = []

with torch.no_grad():
    for batch in train_loader:
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        all_scores.append(anomaly_score(model, x.to(DEVICE)).cpu())

all_scores = torch.cat(all_scores).numpy()

# Set threshold at the 95th percentile of training scores
threshold         = float(np.percentile(all_scores, 95))
predicted_anomaly = all_scores > threshold

print(f"Mean score : {all_scores.mean():.6f}  |  Std : {all_scores.std():.6f}")
print(f"Threshold  : {threshold:.6f}  (95th percentile)")
print(f"Flagged    : {predicted_anomaly.sum()} / {len(predicted_anomaly)} samples")


Mean score : 0.182704  |  Std : 0.019767
Threshold  : 0.221809  (95th percentile)
Flagged    : 50 / 992 samples
